# 🍎 Smart Fridge — Fruit3 Clean Experiment

**Fresh, self-contained experiment. No legacy dataset-merge logic.**

Trains two models:
- **`yolo_fruit3.pt`** — YOLOv8n detector, 3 classes: apple / banana / orange
- **`freshness_fruit3.pt`** — MobileNetV3-Small, 2-class training (Fresh/Spoiled), 3 runtime labels via thresholds

Detection dataset : `mbkinaci/fruit-images-for-object-detection` (Kaggle, Pascal VOC XML)  
Freshness dataset : `swoyam2609/fresh-and-stale-classification` (Kaggle, image folders)

---
### ⚙️ Before running
1. **Runtime → Change runtime type → T4 GPU**
2. Fill in Kaggle credentials below
3. **Runtime → Run all**

---
## 0 · Configuration

In [ ]:
# ── Kaggle credentials ────────────────────────────────────────────────────────
KAGGLE_USERNAME = "YOUR_KAGGLE_USERNAME"   # from kaggle.json
KAGGLE_KEY      = "YOUR_KAGGLE_KEY"

# ── Detector hyper-parameters ─────────────────────────────────────────────────
YOLO_EPOCHS   = 40
YOLO_IMGSZ    = 640
YOLO_BATCH    = 8

# ── Freshness hyper-parameters ────────────────────────────────────────────────
FRESH_EPOCHS  = 30
FRESH_BATCH   = 32
FRESH_LR      = 1e-3

# ── Runtime inference thresholds (2-class model → 3 runtime labels) ───────────
FRESH_THRESH   = 0.75   # P(Fresh)   >= 0.75  →  "Fresh"
SPOILED_THRESH = 0.75   # P(Spoiled) >= 0.75  →  "Spoiled"  (i.e. P(Fresh) <= 0.25)
                         # otherwise           →  "Moderate"

# ── Export paths ──────────────────────────────────────────────────────────────
YOLO_EXPORT   = "/content/yolo_fruit3.pt"
FRESH_EXPORT  = "/content/freshness_fruit3.pt"

# ── Device ────────────────────────────────────────────────────────────────────
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cpu":
    print("⚠️  No GPU — training will be slow. Switch to T4 GPU runtime.")

---
## 1 · Install Dependencies

In [ ]:
%%capture
!pip install ultralytics kaggle --quiet
!pip install torchvision scikit-learn --quiet

In [ ]:
import os, json, shutil, random, math
from pathlib import Path
from xml.etree import ElementTree as ET
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from PIL import Image as PILImage

from ultralytics import YOLO

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")

---
## 2 · Kaggle Credentials

In [ ]:
assert KAGGLE_USERNAME != "YOUR_KAGGLE_USERNAME", \
    "❌ Set KAGGLE_USERNAME and KAGGLE_KEY in cell 0."

kaggle_dir = Path("/root/.kaggle")
kaggle_dir.mkdir(parents=True, exist_ok=True)
with open(kaggle_dir / "kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(kaggle_dir / "kaggle.json", 0o600)
print("Kaggle credentials written ✓")

---
## 3 · Part A — YOLOv8 Detector (apple / banana / orange)

**Dataset:** `mbkinaci/fruit-images-for-object-detection`  
Format: Pascal VOC XML annotations.  
This cell converts them to YOLO `.txt` format, filters to 3 classes, builds `data.yaml`.

### 3.1 · Download Detection Dataset

In [ ]:
DET_RAW = Path("/content/fruit_det_raw")
DET_RAW.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d mbkinaci/fruit-images-for-object-detection \
    -p {DET_RAW} --unzip --quiet

# The dataset unzips to:
#   fruit-images-for-object-detection/
#     train_zip/train/*.jpg  + *.xml
#     test_zip/test/*.jpg   + *.xml
print("Downloaded. Top-level contents:")
for p in sorted(DET_RAW.rglob("*"))[:25]:
    print(" ", p.relative_to(DET_RAW))

### 3.2 · Inspect Raw Class Names

In [ ]:
# Read every XML file and collect all class names actually present
all_xml = sorted(DET_RAW.rglob("*.xml"))
print(f"Total XML annotation files: {len(all_xml)}")

raw_class_counter = Counter()
for xml_path in all_xml:
    tree = ET.parse(xml_path)
    root = tree.getroot()
    for obj in root.findall("object"):
        name = obj.find("name").text.strip().lower()
        raw_class_counter[name] += 1

print("\nRaw class names in dataset:")
for cls, count in raw_class_counter.most_common():
    print(f"  '{cls}' : {count} annotations")

### 3.3 · Convert VOC XML → YOLO TXT (3 classes only)

In [ ]:
# ── Target classes (fixed order — must not change after training) ─────────────
CLASSES     = ["apple", "banana", "orange"]
CLASS_IDX   = {c: i for i, c in enumerate(CLASSES)}   # apple→0, banana→1, orange→2

# Normalise raw XML class names to pipeline class names.
# The mbkinaci dataset uses exactly 'apple', 'banana', 'orange' (already clean).
# This map handles any casing variants just in case.
def normalise_class(raw: str) -> str | None:
    low = raw.lower().strip()
    if "apple"  in low: return "apple"
    if "banana" in low: return "banana"
    if "orange" in low: return "orange"
    return None   # discard everything else


def voc_xml_to_yolo(xml_path: Path, img_dir: Path, lbl_dir: Path) -> int:
    """
    Parse one Pascal VOC XML file and write a YOLO .txt label file.
    Returns number of valid annotations written (0 = file deleted).
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    size  = root.find("size")
    img_w = int(size.find("width").text)
    img_h = int(size.find("height").text)
    if img_w == 0 or img_h == 0:
        return 0

    lines = []
    for obj in root.findall("object"):
        raw_name = obj.find("name").text.strip()
        cls_name = normalise_class(raw_name)
        if cls_name is None:
            continue   # skip classes not in our 3-class set

        bndbox = obj.find("bndbox")
        xmin = float(bndbox.find("xmin").text)
        ymin = float(bndbox.find("ymin").text)
        xmax = float(bndbox.find("xmax").text)
        ymax = float(bndbox.find("ymax").text)

        # Convert to YOLO normalised cx, cy, w, h
        cx = ((xmin + xmax) / 2) / img_w
        cy = ((ymin + ymax) / 2) / img_h
        bw = (xmax - xmin) / img_w
        bh = (ymax - ymin) / img_h

        # Clamp to [0, 1] to handle any annotation imprecision
        cx, cy = min(max(cx, 0.0), 1.0), min(max(cy, 0.0), 1.0)
        bw, bh = min(max(bw, 0.0), 1.0), min(max(bh, 0.0), 1.0)

        lines.append(f"{CLASS_IDX[cls_name]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

    stem = xml_path.stem

    if not lines:
        # No valid annotations → delete the corresponding image too
        for ext in [".jpg", ".jpeg", ".png", ".JPG"]:
            img = img_dir / (stem + ext)
            if img.exists():
                img.unlink()
        return 0

    lbl_dir.mkdir(parents=True, exist_ok=True)
    (lbl_dir / (stem + ".txt")).write_text("\n".join(lines) + "\n")
    return len(lines)


print(f"Target classes: {CLASSES}")
print(f"Class index   : {CLASS_IDX}")

### 3.4 · Build YOLO Dataset (train / val splits)

In [ ]:
import yaml

YOLO_DATA_DIR = Path("/content/fruit3_yolo")
YOLO_YAML     = YOLO_DATA_DIR / "data.yaml"

# ── Locate raw train and test image/xml folders ────────────────────────────────
# mbkinaci unzips to:
#   fruit-images-for-object-detection/train_zip/train/
#   fruit-images-for-object-detection/test_zip/test/
# Robustly search for them.
def find_split_dir(raw_root: Path, keyword: str) -> Path | None:
    for d in sorted(raw_root.rglob("*")):
        if d.is_dir() and keyword in d.name.lower() and not d.name.lower().startswith("."):
            # Must contain at least one image
            if list(d.glob("*.jpg")) or list(d.glob("*.JPG")):
                return d
    return None

raw_train_dir = find_split_dir(DET_RAW, "train")
raw_test_dir  = find_split_dir(DET_RAW, "test")

print(f"Raw train dir : {raw_train_dir}")
print(f"Raw test dir  : {raw_test_dir}")
assert raw_train_dir, "❌ Could not locate train folder — check download."

# ── Create output folder structure ─────────────────────────────────────────────
for split in ["train", "val"]:
    (YOLO_DATA_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (YOLO_DATA_DIR / split / "labels").mkdir(parents=True, exist_ok=True)


def process_split(src_dir: Path, out_split: str, val_fraction: float = 0.0):
    """
    Copy images and convert XML annotations for one split.
    If val_fraction > 0, randomly hold out that fraction as val images.
    Returns (kept_annotations, removed_images) counts.
    """
    xml_files = sorted(src_dir.glob("*.xml"))
    random.seed(42)
    random.shuffle(xml_files)

    kept_ann = 0
    removed  = 0

    for i, xml_path in enumerate(xml_files):
        # Decide which output split this image goes to
        if val_fraction > 0 and i < int(len(xml_files) * val_fraction):
            split_out = "val"
        else:
            split_out = out_split

        img_out_dir = YOLO_DATA_DIR / split_out / "images"
        lbl_out_dir = YOLO_DATA_DIR / split_out / "labels"

        # Convert annotations
        n = voc_xml_to_yolo(xml_path, src_dir, lbl_out_dir)

        if n == 0:
            removed += 1
            continue

        # Copy image
        stem = xml_path.stem
        for ext in [".jpg", ".jpeg", ".png", ".JPG"]:
            src_img = src_dir / (stem + ext)
            if src_img.exists():
                shutil.copy(str(src_img), str(img_out_dir / (stem + ext)))
                break

        kept_ann += n

    return kept_ann, removed


# Train folder: hold out 15% as validation
print("\nProcessing train folder (85% train / 15% val split)...")
train_ann, train_removed = process_split(raw_train_dir, "train", val_fraction=0.15)

# Test folder → add to val (gives a richer validation set)
if raw_test_dir:
    print("Processing test folder → adding to val...")
    test_ann, test_removed = process_split(raw_test_dir, "val", val_fraction=0.0)
else:
    test_ann = test_removed = 0

print(f"\nImages removed (zero valid annotations after filtering): "
      f"{train_removed + test_removed}")

# ── Write data.yaml ────────────────────────────────────────────────────────────
cfg = {
    "path":  str(YOLO_DATA_DIR),
    "train": "train/images",
    "val":   "val/images",
    "nc":    len(CLASSES),
    "names": CLASSES,
}
with open(YOLO_YAML, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"\ndata.yaml written → {YOLO_YAML}")
print(f"  nc    : {cfg['nc']}")
print(f"  names : {cfg['names']}")
for split in ["train", "val"]:
    n_imgs = len(list((YOLO_DATA_DIR / split / "images").glob("*")))
    n_lbls = len(list((YOLO_DATA_DIR / split / "labels").glob("*.txt")))
    print(f"  {split:<6}: {n_imgs} images, {n_lbls} label files")

### 3.5 · Per-Class Annotation Counts (pre-training check)

In [ ]:
def count_annotations(data_dir: Path, split: str) -> Counter:
    """Count per-class annotations in a split's labels folder."""
    cnt = Counter()
    lbl_dir = data_dir / split / "labels"
    if not lbl_dir.exists():
        return cnt
    for f in lbl_dir.glob("*.txt"):
        for line in f.read_text().splitlines():
            parts = line.strip().split()
            if parts:
                cnt[int(parts[0])] += 1
    return cnt

print("=" * 50)
print("PRE-TRAINING ANNOTATION COUNTS")
print("=" * 50)
for split in ["train", "val"]:
    cnt = count_annotations(YOLO_DATA_DIR, split)
    total = sum(cnt.values())
    print(f"\n  {split.upper()} split ({total} total annotations):")
    print(f"  {'Class':<10} {'ID':>4}  {'Count':>8}")
    print(f"  {'-'*26}")
    for cls_name in CLASSES:
        idx = CLASS_IDX[cls_name]
        print(f"  {cls_name:<10} {idx:>4}  {cnt.get(idx, 0):>8}")

    # Warn if any class is missing
    missing = [c for c in CLASSES if cnt.get(CLASS_IDX[c], 0) == 0]
    if missing:
        print(f"  ⚠️  MISSING in {split}: {missing}")
    else:
        print(f"  ✅ All 3 classes present in {split} split")

print("\n" + "=" * 50)
print("Proceed to training only if all 3 classes show > 0 above.")
print("=" * 50)

### 3.6 · Visualise Sample Annotations

In [ ]:
CLASS_COLORS = {"apple": "#e74c3c", "banana": "#f1c40f", "orange": "#e67e22"}

def draw_yolo_boxes(img_path: Path, lbl_path: Path) -> np.ndarray:
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 5: continue
            cls_id = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])
            x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
            x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
            cls_name = CLASSES[cls_id] if cls_id < len(CLASSES) else str(cls_id)
            color_hex = CLASS_COLORS.get(cls_name, "#ffffff")
            color_rgb = tuple(int(color_hex.lstrip("#")[i:i+2], 16) for i in (0, 2, 4))
            cv2.rectangle(img, (x1, y1), (x2, y2), color_rgb, 2)
            cv2.putText(img, cls_name, (x1, max(y1-6, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_rgb, 2)
    return img

train_imgs = sorted((YOLO_DATA_DIR / "train" / "images").glob("*.jpg"))[:6]
n = len(train_imgs)
if n > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, img_path in zip(axes.flat, train_imgs):
        lbl_path = YOLO_DATA_DIR / "train" / "labels" / img_path.with_suffix(".txt").name
        ax.imshow(draw_yolo_boxes(img_path, lbl_path))
        ax.axis("off")
        ax.set_title(img_path.name, fontsize=8)
    # Hide unused axes
    for ax in axes.flat[n:]:
        ax.axis("off")
    plt.suptitle("Sample Training Annotations — apple / banana / orange", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No training images found — check dataset download.")

### 3.7 · Train YOLOv8n Detector

In [ ]:
yolo_model = YOLO("yolov8n.pt")   # COCO-pretrained nano — fine-tune on our 3 classes

print(f"Training YOLOv8n on {YOLO_DATA_DIR.name}")
print(f"  Classes  : {CLASSES}")
print(f"  Epochs   : {YOLO_EPOCHS}")
print(f"  Img size : {YOLO_IMGSZ}")
print(f"  Batch    : {YOLO_BATCH}")
print(f"  Device   : {DEVICE}")

In [ ]:
results = yolo_model.train(
    data        = str(YOLO_YAML),
    epochs      = YOLO_EPOCHS,
    imgsz       = YOLO_IMGSZ,
    batch       = YOLO_BATCH,
    device      = DEVICE,
    project     = "runs/detect",
    name        = "fruit3",
    # Augmentations — fruit colours matter, use moderate colour jitter
    hsv_h       = 0.015,
    hsv_s       = 0.5,
    hsv_v       = 0.3,
    degrees     = 5.0,
    fliplr      = 0.5,
    mosaic      = 1.0,
    mixup       = 0.05,
    # Stability
    save        = True,
    save_period = 10,
    patience    = 15,
    verbose     = True,
)

best_pt = Path("runs/detect/fruit3/weights/best.pt")
assert best_pt.exists(), f"best.pt not found at {best_pt}"
shutil.copy(str(best_pt), YOLO_EXPORT)
print(f"\n✅ Detector saved → {YOLO_EXPORT}")

### 3.8 · Evaluate Detector

In [ ]:
best_yolo = YOLO(YOLO_EXPORT)
val_metrics = best_yolo.val(data=str(YOLO_YAML), device=DEVICE, verbose=False)

print(f"mAP@50     : {val_metrics.box.map50:.4f}")
print(f"mAP@50-95  : {val_metrics.box.map:.4f}")
print(f"Precision  : {val_metrics.box.mp:.4f}")
print(f"Recall     : {val_metrics.box.mr:.4f}")

# Training curves
results_csv = Path("runs/detect/fruit3/results.csv")
if results_csv.exists():
    import pandas as pd
    df = pd.read_csv(results_csv); df.columns = df.columns.str.strip()
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(df["epoch"], df["train/box_loss"], label="train", color="royalblue")
    axes[0].plot(df["epoch"], df["val/box_loss"],   label="val",   color="tomato", linestyle="--")
    axes[0].set_title("Box Loss"); axes[0].legend()
    axes[1].plot(df["epoch"], df["metrics/mAP50(B)"],    color="seagreen")
    axes[1].set_title("mAP@50")
    axes[2].plot(df["epoch"], df["metrics/mAP50-95(B)"], color="darkorange")
    axes[2].set_title("mAP@50-95")
    for ax in axes: ax.set_xlabel("Epoch"); ax.grid(alpha=0.3)
    plt.suptitle("YOLOv8n Training Curves — Fruit3", fontsize=13)
    plt.tight_layout(); plt.show()

### 3.9 · Test Detector on a Sample Image

In [ ]:
# Run the trained detector on a validation image and display results
val_imgs = sorted((YOLO_DATA_DIR / "val" / "images").glob("*.jpg"))
assert val_imgs, "No val images found"

# Pick first image that has at least one annotation
sample_img = None
for img_path in val_imgs:
    lbl = YOLO_DATA_DIR / "val" / "labels" / img_path.with_suffix(".txt").name
    if lbl.exists() and lbl.stat().st_size > 0:
        sample_img = img_path
        break
sample_img = sample_img or val_imgs[0]

frame = cv2.imread(str(sample_img))
preds = best_yolo.predict(frame, conf=0.25, device="cpu", verbose=False)[0]

print(f"Sample image : {sample_img.name}")
print(f"Detections   : {len(preds.boxes)}")
print(f"{'Class':<10} {'Conf':>6}  {'BBox (x1,y1,x2,y2)'}")
print("-" * 55)
for box in preds.boxes:
    cls_name = best_yolo.names[int(box.cls[0])]
    conf     = float(box.conf[0])
    coords   = tuple(map(int, box.xyxy[0].tolist()))
    print(f"  {cls_name:<10} {conf:>6.3f}  {coords}")

# Save annotated prediction image
PRED_OUT = Path("/content/fruit3_prediction.jpg")
annotated = preds.plot()
cv2.imwrite(str(PRED_OUT), annotated)
print(f"\nAnnotated prediction saved → {PRED_OUT}")

# Display inline
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title("Predictions (conf≥0.25)"); axes[1].axis("off")
plt.suptitle(f"Detector test — {sample_img.name}", fontsize=12)
plt.tight_layout(); plt.show()

---
## 4 · Part B — MobileNetV3 Freshness Classifier

**Dataset:** `swoyam2609/fresh-and-stale-classification`  
**Training:** 2-class (Fresh / Spoiled)  
**Runtime:** 3 labels via confidence thresholds  

Only apple, banana, orange folders are used. Other produce (cucumber, potato, etc.) is discarded.

### 4.1 · Download Freshness Dataset

In [ ]:
FRESH_RAW = Path("/content/fresh_stale_raw")
FRESH_RAW.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d swoyam2609/fresh-and-stale-classification \
    -p {FRESH_RAW} --unzip --quiet

print("Downloaded. Folder names (first 40):")
for p in sorted(FRESH_RAW.rglob("*"))[:40]:
    if p.is_dir():
        print(" ", p.relative_to(FRESH_RAW))

### 4.2 · Remap to 2-Class (Fresh / Spoiled) — apple, banana, orange only

In [ ]:
# ── Which source folders map to Fresh or Spoiled ──────────────────────────────
# The dataset has folders like: freshapples, freshbanana, freshoranges,
#                                rottenapples, rottenbanana, rottenoranges, ...
# We keep ONLY apple, banana, orange and discard everything else.

KEEP_FRUITS = {"apple", "banana", "orange"}   # strictly 3

# Label map: source folder name (lowercase) → training label
# Built dynamically so it handles minor naming variations in the dataset.
def folder_to_label(folder_name: str) -> tuple[str, str] | None:
    """
    Returns (fruit_name, label) or None if folder should be discarded.
    Label is 'Fresh' or 'Spoiled'.
    """
    low = folder_name.lower().strip()

    # Determine freshness direction
    if low.startswith("fresh"):
        direction = "Fresh"
        remainder = low[len("fresh"):].strip("_- ")
    elif low.startswith("rotten") or low.startswith("stale") or low.startswith("spoiled"):
        direction = "Spoiled"
        for prefix in ("rotten", "stale", "spoiled"):
            if low.startswith(prefix):
                remainder = low[len(prefix):].strip("_- ")
                break
    else:
        return None   # unrecognised prefix

    # Determine which fruit
    if "apple"  in remainder: fruit = "apple"
    elif "banana" in remainder: fruit = "banana"
    elif "orange" in remainder: fruit = "orange"
    else:
        return None   # not one of our 3 fruits → discard

    return fruit, direction


# ── Build the remapping table ──────────────────────────────────────────────────
# Find all leaf directories that contain images
source_dirs = {}
for d in sorted(FRESH_RAW.rglob("*")):
    if d.is_dir() and list(d.glob("*.jpg")) + list(d.glob("*.png")) + list(d.glob("*.jpeg")):
        source_dirs[d.name.lower()] = d

print(f"Image-containing folders found: {len(source_dirs)}")
print("\nFolder → (fruit, label) mapping:")
label_plan = {}   # folder_name → (fruit, direction)
for folder_name, folder_path in sorted(source_dirs.items()):
    result = folder_to_label(folder_name)
    n_imgs = len(list(folder_path.glob("*.jpg")) + list(folder_path.glob("*.png")) +
                 list(folder_path.glob("*.jpeg")))
    if result:
        label_plan[folder_name] = (result, folder_path, n_imgs)
        print(f"  ✅ {folder_name:<25} → fruit={result[0]:<8} label={result[1]:<8} ({n_imgs} imgs)")
    else:
        print(f"  ⬛ {folder_name:<25} → discarded")

In [ ]:
# ── Build remapped dataset (Fresh / Spoiled only, train/val/test split) ────────
FRESH_DATA_DIR = Path("/content/freshness_fruit3")
SPLITS = {"train": 0.80, "val": 0.10, "test": 0.10}
FRESH_LABELS = ["Fresh", "Spoiled"]

for split in SPLITS:
    for label in FRESH_LABELS:
        (FRESH_DATA_DIR / split / label).mkdir(parents=True, exist_ok=True)

copied = {"Fresh": 0, "Spoiled": 0}
random.seed(42)

for folder_name, ((fruit, direction), folder_path, _) in label_plan.items():
    images = (sorted(folder_path.glob("*.jpg")) +
              sorted(folder_path.glob("*.jpeg")) +
              sorted(folder_path.glob("*.png")))
    random.shuffle(images)
    n = len(images)
    if n == 0:
        continue

    n_train = int(n * SPLITS["train"])
    n_val   = int(n * SPLITS["val"])
    assignments = (
        [("train", img) for img in images[:n_train]] +
        [("val",   img) for img in images[n_train:n_train + n_val]] +
        [("test",  img) for img in images[n_train + n_val:]]
    )

    for split_name, img_path in assignments:
        dst = FRESH_DATA_DIR / split_name / direction / f"{folder_name}_{img_path.name}"
        shutil.copy(str(img_path), str(dst))

    copied[direction] += n

print("Images copied per label (2-class training dataset):")
for lbl in FRESH_LABELS:
    print(f"  {lbl:<10}: {copied[lbl]}")

print("\nNote: 'Moderate' is NOT a training class.")
print("It is derived at inference time from P(Fresh) thresholds.")
print("\nPer split:")
for split in SPLITS:
    for lbl in FRESH_LABELS:
        d = FRESH_DATA_DIR / split / lbl
        n = len(list(d.glob("*"))) if d.exists() else 0
        print(f"  {split}/{lbl}: {n} images")

### 4.3 · Build DataLoaders

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.08),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(str(FRESH_DATA_DIR / "train"), transform=train_tf)
val_ds   = datasets.ImageFolder(str(FRESH_DATA_DIR / "val"),   transform=val_tf)
test_ds  = datasets.ImageFolder(str(FRESH_DATA_DIR / "test"),  transform=val_tf)

# ImageFolder sorts alphabetically: Fresh=0, Spoiled=1
assert train_ds.classes == ["Fresh", "Spoiled"], (
    f"Expected ['Fresh','Spoiled'], got {train_ds.classes}. "
    "Check directory names inside freshness_fruit3/train/"
)

train_dl = DataLoader(train_ds, batch_size=FRESH_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=FRESH_BATCH, shuffle=False, num_workers=2)
test_dl  = DataLoader(test_ds,  batch_size=FRESH_BATCH, shuffle=False, num_workers=2)

# Class-weighted loss to handle any imbalance between Fresh and Spoiled
class_counts = Counter(train_ds.targets)
n_total = sum(class_counts.values())
class_weights = torch.tensor(
    [n_total / (2 * class_counts[i]) for i in range(2)],
    dtype=torch.float32
).to(DEVICE)

print(f"Train : {len(train_ds)} images | {train_ds.classes}")
print(f"Val   : {len(val_ds)} images")
print(f"Test  : {len(test_ds)} images")
print(f"Class weights (Fresh, Spoiled): {class_weights.tolist()}")

### 4.4 · Define MobileNetV3-Small Model

In [ ]:
# 2-class output head (Fresh=0, Spoiled=1)
# Pretrained ImageNet weights → fine-tune all layers
backbone = models.mobilenet_v3_small(
    weights=models.MobileNet_V3_Small_Weights.DEFAULT
)
in_features = backbone.classifier[3].in_features
backbone.classifier[3] = nn.Linear(in_features, 2)
model = backbone.to(DEVICE)

total_params   = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"MobileNetV3-Small  total params    : {total_params:,}")
print(f"                   trainable params: {trainable_params:,}")

### 4.5 · Train Freshness Classifier

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=FRESH_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=FRESH_LR,
    steps_per_epoch=len(train_dl), epochs=FRESH_EPOCHS, pct_start=0.2,
)

FRESH_PT     = Path(FRESH_EXPORT)
best_val_acc = 0.0
history      = {"train_loss": [], "val_loss": [], "val_acc": []}

print(f"Training MobileNetV3-Small for {FRESH_EPOCHS} epochs on {DEVICE}...\n")

for epoch in range(1, FRESH_EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * imgs.size(0)
    train_loss = running_loss / len(train_ds)

    # ── Validate ───────────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0; correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            val_loss += criterion(out, labels).item() * imgs.size(0)
            correct  += (out.argmax(1) == labels).sum().item()
            total    += labels.size(0)
    val_loss /= len(val_ds)
    val_acc   = correct / total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    flag = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), str(FRESH_PT))
        flag = "  ← best ✓"

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{FRESH_EPOCHS}  "
              f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
              f"val_acc={val_acc:.4f}{flag}")

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")
print(f"Model saved → {FRESH_PT}")

### 4.6 · Evaluate Freshness Classifier on Test Set

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

model.load_state_dict(torch.load(str(FRESH_PT), map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_dl:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print("Test set — 2-class evaluation (Fresh / Spoiled):")
print(classification_report(all_labels, all_preds,
                             target_names=["Fresh", "Spoiled"]))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Fresh", "Spoiled"])
ax.set_yticklabels(["Fresh", "Spoiled"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Freshness Classifier")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im); plt.tight_layout(); plt.show()

---
## 5 · Inference Helpers

These functions implement the full pipeline:
detect → crop → classify freshness → estimate shelf life → alert

### 5.1 · Freshness Inference (2-class model → 3 runtime labels)

In [ ]:
# ── Pre-processing (mirrors what the DataLoader does at test time) ─────────────
_infer_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def preprocess_crop(crop_bgr: np.ndarray) -> torch.Tensor:
    """BGR numpy array → (1, 3, 224, 224) normalised tensor."""
    rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    pil = PILImage.fromarray(rgb)
    return _infer_tf(pil).unsqueeze(0)   # (1, 3, 224, 224)


def classify_freshness(crop_bgr: np.ndarray) -> dict:
    """
    Run the freshness model on a single BGR crop.

    Returns
    -------
    dict:
        freshness_label : 'Fresh' | 'Moderate' | 'Spoiled'
        freshness_score : float = P(Fresh)  [0=certainly spoiled, 1=certainly fresh]
        probabilities   : {'Fresh': float, 'Spoiled': float}

    Label derivation (threshold-based, no synthetic training class):
        P(Fresh)   >= FRESH_THRESH (0.75)   →  Fresh
        P(Spoiled) >= SPOILED_THRESH (0.75) →  Spoiled    [P(Fresh) <= 0.25]
        otherwise                           →  Moderate   [0.25 < P(Fresh) < 0.75]
    """
    model.eval()
    tensor = preprocess_crop(crop_bgr).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1).squeeze(0)  # (2,)
    fresh_p   = float(probs[0])   # index 0 = Fresh (alphabetical sort)
    spoiled_p = float(probs[1])   # index 1 = Spoiled

    if fresh_p >= FRESH_THRESH:
        label = "Fresh"
    elif spoiled_p >= SPOILED_THRESH:
        label = "Spoiled"
    else:
        label = "Moderate"

    return {
        "freshness_label": label,
        "freshness_score": round(fresh_p, 4),
        "probabilities":   {"Fresh": round(fresh_p, 4), "Spoiled": round(spoiled_p, 4)},
    }

### 5.2 · Shelf-Life Estimator

In [ ]:
# ── Baseline shelf life (days at ~4 °C) ───────────────────────────────────────
BASELINE_SHELF_LIFE = {
    "apple":   42,   # refrigerated, unwaxed
    "banana":   5,   # ripens quickly even when cold
    "orange":  21,   # refrigerated
    "unknown":  5,   # conservative fallback
}

def _score_to_modifier(score: float) -> float:
    """
    Map freshness_score (P(Fresh), 0→1) to a shelf-life multiplier (0→1).
    Piecewise linear: heavily penalises low scores, close to 1.0 near-peak fresh.
    """
    score = min(max(score, 0.0), 1.0)
    breakpoints = [
        (0.00, 0.00),
        (0.25, 0.05),   # Spoiled threshold
        (0.50, 0.30),
        (0.65, 0.55),
        (0.75, 0.75),   # Fresh threshold
        (1.00, 1.00),
    ]
    for i in range(1, len(breakpoints)):
        s0, m0 = breakpoints[i - 1]
        s1, m1 = breakpoints[i]
        if s0 <= score <= s1:
            t = (score - s0) / (s1 - s0)
            return m0 + t * (m1 - m0)
    return 1.0


def estimate_shelf_life(
    item_name: str,
    freshness_score: float,
    freshness_label: str,
) -> int:
    """
    Estimate days remaining for a detected fruit.

    Parameters
    ----------
    item_name       : 'apple' | 'banana' | 'orange' | 'unknown'
    freshness_score : P(Fresh) from classify_freshness(), in [0, 1]
    freshness_label : 'Fresh' | 'Moderate' | 'Spoiled'

    Returns
    -------
    int — estimated days remaining (0 if spoiled or score < 0.10)
    """
    if freshness_label == "Spoiled" or freshness_score < 0.10:
        return 0
    baseline  = BASELINE_SHELF_LIFE.get(item_name.lower(), BASELINE_SHELF_LIFE["unknown"])
    modifier  = _score_to_modifier(freshness_score)
    return max(0, math.floor(baseline * modifier))


# Quick smoke test
print(f"{'Item':<10} {'Score':>6} {'Label':<10} {'Days':>6}")
print("-" * 36)
tests = [
    ("apple",  0.92, "Fresh"),
    ("apple",  0.55, "Moderate"),
    ("banana", 0.20, "Spoiled"),
    ("orange", 0.78, "Fresh"),
    ("orange", 0.40, "Moderate"),
    ("banana", 0.80, "Fresh"),
]
for item, score, label in tests:
    days = estimate_shelf_life(item, score, label)
    print(f"{item:<10} {score:>6.2f} {label:<10} {days:>6}")

### 5.3 · Alert Generator

In [ ]:
EXPIRY_THRESHOLD = 2   # days

def generate_alerts(results: list[dict]) -> list[str]:
    """
    Given a list of per-item result dicts, return a list of alert strings.
    Each dict must have: class_name, freshness_label, freshness_score, days_remaining.
    """
    alerts = []
    for item in results:
        name  = item["class_name"]
        label = item["freshness_label"]
        days  = item["days_remaining"]

        if label == "Spoiled":
            alerts.append(f"🔴 SPOILED  — {name}: discard immediately")
        elif days <= EXPIRY_THRESHOLD:
            alerts.append(f"🟡 EXPIRING — {name}: {days} day(s) left, use soon")
        elif label == "Moderate":
            alerts.append(f"🟠 MODERATE — {name}: freshness declining ({days}d left)")
        else:
            alerts.append(f"🟢 FRESH    — {name}: {days} day(s) remaining")
    return alerts

---
## 6 · End-to-End Pipeline Test

Runs the full chain on a sample image:  
**detect → crop → classify freshness → estimate shelf life → alert**

In [ ]:
# Load trained weights (in case cells were run out of order)
best_yolo = YOLO(YOLO_EXPORT)
model.load_state_dict(torch.load(str(FRESH_PT), map_location=DEVICE))
model.eval()

# ── Pick a test image ─────────────────────────────────────────────────────────
# Uses the same val image from section 3.9, or you can replace this path.
test_image_path = sample_img   # set to any Path or string to test a different image
# test_image_path = "/content/my_fruit_photo.jpg"  # ← uncomment and change

frame = cv2.imread(str(test_image_path))
h, w  = frame.shape[:2]
print(f"Image: {Path(test_image_path).name}  ({w}×{h})")
print("=" * 60)

# ── Step 1: Detect ────────────────────────────────────────────────────────────
det_results = best_yolo.predict(frame, conf=0.30, device="cpu", verbose=False)[0]
print(f"Detections: {len(det_results.boxes)}")

pipeline_results = []
PAD = 8

for box in det_results.boxes:
    cls_name = best_yolo.names[int(box.cls[0])]
    conf_val = float(box.conf[0])
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

    # ── Step 2: Crop (with padding) ───────────────────────────────────────────
    cx1, cy1 = max(0, x1 - PAD), max(0, y1 - PAD)
    cx2, cy2 = min(w, x2 + PAD), min(h, y2 + PAD)
    crop = frame[cy1:cy2, cx1:cx2]

    if crop.size == 0:
        print(f"  {cls_name}: empty crop, skipping")
        continue

    # ── Step 3: Classify freshness ────────────────────────────────────────────
    fresh_result = classify_freshness(crop)

    # ── Step 4: Estimate shelf life ───────────────────────────────────────────
    days = estimate_shelf_life(
        cls_name,
        fresh_result["freshness_score"],
        fresh_result["freshness_label"],
    )

    result = {
        "class_name":      cls_name,
        "det_confidence":  round(conf_val, 3),
        "freshness_label": fresh_result["freshness_label"],
        "freshness_score": fresh_result["freshness_score"],
        "probabilities":   fresh_result["probabilities"],
        "days_remaining":  days,
        "bbox":            (x1, y1, x2, y2),
    }
    pipeline_results.append(result)

    print(f"  {cls_name:<8} det_conf={conf_val:.2f}  "
          f"freshness={fresh_result['freshness_label']:<10}  "
          f"score={fresh_result['freshness_score']:.3f}  "
          f"days={days}")

# ── Step 5: Alerts ────────────────────────────────────────────────────────────
alerts = generate_alerts(pipeline_results)
print("\nAlerts:")
for alert in alerts:
    print(" ", alert)

if not alerts:
    print("  (no detections or no alerts triggered)")

# ── Visualise ─────────────────────────────────────────────────────────────────
VIS_COLORS = {"Fresh": (0, 200, 0), "Moderate": (0, 165, 255), "Spoiled": (0, 0, 220)}
vis = frame.copy()
for r in pipeline_results:
    x1, y1, x2, y2 = r["bbox"]
    color = VIS_COLORS.get(r["freshness_label"], (200, 200, 200))
    cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
    tag = f"{r['class_name']} {r['freshness_label']} {r['days_remaining']}d"
    cv2.putText(vis, tag, (x1, max(y1 - 6, 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

E2E_OUT = Path("/content/fruit3_e2e_result.jpg")
cv2.imwrite(str(E2E_OUT), vis)
print(f"\nEnd-to-end result image saved → {E2E_OUT}")

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("End-to-End Pipeline Result (Green=Fresh, Orange=Moderate, Red=Spoiled)")
plt.tight_layout()
plt.show()

---
## 7 · Download Trained Models

In [ ]:
from google.colab import files

for pt in [YOLO_EXPORT, FRESH_EXPORT]:
    p = Path(pt)
    if p.exists():
        size_mb = p.stat().st_size / 1e6
        print(f"  {p.name:<30} {size_mb:.1f} MB")
    else:
        print(f"  {p.name:<30} ← NOT FOUND (training may not have completed)")

print("\nDownloading yolo_fruit3.pt ...")
files.download(YOLO_EXPORT)
print("Downloading freshness_fruit3.pt ...")
files.download(FRESH_EXPORT)

In [ ]:
# Optional: save to Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy(YOLO_EXPORT,  '/content/drive/MyDrive/smart_fridge/yolo_fruit3.pt')
# shutil.copy(FRESH_EXPORT, '/content/drive/MyDrive/smart_fridge/freshness_fruit3.pt')
# print('Saved to Google Drive ✓')

---
## 8 · Summary

| Artifact | Architecture | Training | Runtime output |
|---|---|---|---|
| `yolo_fruit3.pt` | YOLOv8n | 3-class: apple / banana / orange | Bounding boxes + class labels |
| `freshness_fruit3.pt` | MobileNetV3-Small | **2-class**: Fresh / Spoiled | **3 runtime labels** via thresholds |

### How Moderate is derived (no training class needed)

```
freshness_score = P(Fresh)          # 0.0 = certainly spoiled, 1.0 = certainly fresh

if P(Fresh)   >= 0.75  →  Fresh
if P(Spoiled) >= 0.75  →  Spoiled   # equivalent: P(Fresh) <= 0.25
otherwise              →  Moderate  # 0.25 < P(Fresh) < 0.75
```

### Shelf life baselines (days at ~4 °C)

| Fruit | Baseline |
|---|---|
| apple | 42 days |
| banana | 5 days |
| orange | 21 days |
| unknown | 5 days |

### To run on a new image

```python
# Load models (if starting fresh)
best_yolo = YOLO("/content/yolo_fruit3.pt")
model.load_state_dict(torch.load("/content/freshness_fruit3.pt", map_location="cpu"))
model.eval()

# Point at your image and run section 6 again
test_image_path = "/content/my_photo.jpg"
```

### Deploy on Raspberry Pi

```bash
scp yolo_fruit3.pt freshness_fruit3.pt pi@<pi_ip>:~/smart_fridge/
python3 pipeline.py --yolo yolo_fruit3.pt --freshness freshness_fruit3.pt --source 0
```